# OSM Download from Polygon
Download OSM layers (roads, buildings, amenities, land use, green space) for a given polygon boundary.

## 1. Install & Import

In [ ]:
# !pip install osmnx geopandas shapely

In [ ]:
import re
import traceback
from pathlib import Path

import geopandas as gpd
import osmnx as ox

## 2. Settings — Edit Here

In [ ]:
# ── Input shapefile ────────────────────────────────────────────────────────
FUA_SHP      = r"D:\000_SCI\10_Compact_city\3_FUA_reference\GHS_FUA_cities_subset_clean.shp"
CITY_COL     = "eFUA_name"          # column that holds the city name

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_ROOT  = r"D:\000_SCI\10_Compact_city\OSM_data"

# ── Road network type ──────────────────────────────────────────────────────
# Options: "drive" | "walk" | "bike" | "all"
NETWORK_TYPE = "drive"

## 3. Load Shapefile

In [ ]:
fua = gpd.read_file(FUA_SHP)

# Reproject to EPSG:4326 (required by osmnx)
if fua.crs is None or fua.crs.to_epsg() != 4326:
    fua = fua.to_crs(epsg=4326)

print(f"Loaded {len(fua)} cities")
print(f"Columns: {list(fua.columns)}")
fua[[CITY_COL, 'geometry']].head()

## 4. Helper — Clean Before Saving

`ox.features_from_polygon` returns a MultiIndex `(element_type, osmid)`.  
When pyogrio writes this to a GeoPackage it tries to add a field called **`FID`**, which is reserved by the GPKG format and causes a `FieldError`.  
The helper below fixes that before every `.to_file()` call.

In [ ]:
def _clean_for_save(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Prepare an osmnx GeoDataFrame for writing to GeoPackage.

    Fixes applied:
    1. Reset MultiIndex (element_type, osmid) → plain columns.
    2. Rename any column called 'FID' → 'osm_fid'  (reserved by GPKG/OGR).
    3. Convert all column names to plain strings.
    4. Drop duplicate column names (keep first occurrence).
    """
    gdf = gdf.reset_index()
    gdf.columns = [str(c) for c in gdf.columns]
    if "FID" in gdf.columns:
        gdf = gdf.rename(columns={"FID": "osm_fid"})
    gdf = gdf.loc[:, ~gdf.columns.duplicated()]
    return gdf

## 5. Download Functions + Batch Loop

In [ ]:
# ── Download functions ─────────────────────────────────────────────────────

def download_roads(polygon, output_path, network_type=NETWORK_TYPE):
    """Road network edges."""
    G = ox.graph_from_polygon(polygon, network_type=network_type)
    _, edges = ox.graph_to_gdfs(G)
    edges = _clean_for_save(edges)
    edges.to_file(output_path, layer="roads", driver="GPKG")
    print(f"  roads      : {len(edges):,} segments")


def download_buildings(polygon, output_path):
    """Building footprints clipped to the polygon."""
    gdf = ox.features_from_polygon(polygon, tags={"building": True})
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf = gdf.clip(polygon)
    gdf = _clean_for_save(gdf)
    gdf.to_file(output_path, layer="buildings", driver="GPKG")
    print(f"  buildings  : {len(gdf):,} footprints")


def download_amenities(polygon, output_path):
    """All OSM amenity features (schools, hospitals, restaurants, etc.)."""
    gdf = ox.features_from_polygon(polygon, tags={"amenity": True})
    gdf = gdf.clip(polygon)
    gdf = _clean_for_save(gdf)
    gdf.to_file(output_path, layer="amenities", driver="GPKG")
    print(f"  amenities  : {len(gdf):,} features")


def download_landuse(polygon, output_path):
    """Land-use polygons."""
    gdf = ox.features_from_polygon(polygon, tags={"landuse": True})
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf = _clean_for_save(gdf)
    gdf.to_file(output_path, layer="landuse", driver="GPKG")
    print(f"  landuse    : {len(gdf):,} polygons")


def download_greenspace(polygon, output_path):
    """Parks, forests, and green areas."""
    tags = {
        "leisure": ["park", "garden", "nature_reserve", "recreation_ground"],
        "landuse": ["forest", "grass", "meadow", "orchard", "village_green"],
        "natural": ["wood", "scrub", "heath", "grassland"],
    }
    gdf = ox.features_from_polygon(polygon, tags=tags)
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf = _clean_for_save(gdf)
    gdf.to_file(output_path, layer="greenspace", driver="GPKG")
    print(f"  greenspace : {len(gdf):,} polygons")


# ── Batch loop ─────────────────────────────────────────────────────────────

output_root = Path(OUTPUT_ROOT)
total, failed = len(fua), []

for i, row in fua.iterrows():
    city_name = str(row[CITY_COL])
    safe_name = re.sub(r'[\\/:*?"<>|]', "_", city_name).strip()
    city_dir  = output_root / safe_name
    gpkg      = city_dir / f"{safe_name}_osm.gpkg"

    print(f"[{int(i)+1}/{total}] {city_name}")

    # Skip if the file already exists
    if gpkg.exists():
        print("  Already exists — skipping.")
        continue

    polygon = row.geometry
    if polygon is None or polygon.is_empty:
        print("  Empty geometry — skipping.")
        continue

    city_dir.mkdir(parents=True, exist_ok=True)

    try:
        download_roads(polygon,      gpkg)
        download_buildings(polygon,  gpkg)
        download_amenities(polygon,  gpkg)
        download_landuse(polygon,    gpkg)
        download_greenspace(polygon, gpkg)
        print(f"  Saved → {gpkg}")
    except Exception:
        print(f"  FAILED:\n{traceback.format_exc()}")
        failed.append(city_name)

print("\n" + "="*50)
print(f"Done: {total - len(failed)}/{total} cities succeeded.")
if failed:
    print("Failed:", failed)

In [ ]:
SINGLE_CITY = "Seoul"   # ← change to any city name in the shapefile

row      = fua[fua[CITY_COL] == SINGLE_CITY].iloc[0]
polygon  = row.geometry
city_dir = Path(OUTPUT_ROOT) / SINGLE_CITY
gpkg     = city_dir / f"{SINGLE_CITY}_osm.gpkg"
city_dir.mkdir(parents=True, exist_ok=True)

download_roads(polygon,      gpkg)
download_buildings(polygon,  gpkg)
download_amenities(polygon,  gpkg)
download_landuse(polygon,    gpkg)
download_greenspace(polygon, gpkg)

print("Saved →", gpkg)

## 8. Preview — Amenities with Polygon Boundary

## 7. Preview — Amenities + Boundary for All Downloaded Cities

In [ ]:
import matplotlib.pyplot as plt

output_root = Path(OUTPUT_ROOT)
save_path   = output_root / "all_cities_amenities.png"

# Collect cities that have already been downloaded
downloaded = []
for _, row in fua.iterrows():
    city_name = str(row[CITY_COL])
    safe_name = re.sub(r'[\\/:*?"<>|]', "_", city_name).strip()
    gpkg      = output_root / safe_name / f"{safe_name}_osm.gpkg"
    if gpkg.exists():
        downloaded.append((city_name, safe_name, gpkg, row.geometry))

print(f"Found {len(downloaded)} downloaded cities: {[c[0] for c in downloaded]}")

# One subplot per city
ncols = 3
nrows = -(-len(downloaded) // ncols)   # ceiling division
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = axes.flatten() if len(downloaded) > 1 else [axes]

for ax, (city_name, safe_name, gpkg, boundary_geom) in zip(axes, downloaded):
    try:
        amenities = gpd.read_file(gpkg, layer="amenities")
        boundary  = gpd.GeoDataFrame(geometry=[boundary_geom], crs="EPSG:4326")

        pts  = amenities[amenities.geometry.geom_type == "Point"]
        poly = amenities[amenities.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]

        boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=2, zorder=3)
        if not poly.empty:
            poly.plot(ax=ax, color="orange", alpha=0.5, zorder=2)
        if not pts.empty:
            pts.plot(ax=ax, color="red", markersize=2, alpha=0.6, zorder=4)

        ax.set_title(f"{city_name}\n({len(amenities):,} amenities)", fontsize=11)
    except Exception as e:
        ax.set_title(f"{city_name}\n(error: {e})", fontsize=9)
    ax.set_axis_off()

# Hide unused subplots
for ax in axes[len(downloaded):]:
    ax.set_visible(False)

plt.suptitle("Amenities by City", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()